# DEST — Anexo #5: Batch ablation 64 vs 128 vs 256 (STANDALONE)

In [ ]:
# 0. Setup standalone
import os, sys, subprocess
print("🔧 Setup...")
if os.path.exists("DEST"): subprocess.call(["rm","-rf","DEST"])
subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
subprocess.check_call([sys.executable,"-m","pip","install","-e","DEST","-q"])
if "DEST/src" not in sys.path: sys.path.insert(0,"DEST/src")
import dest
sys.modules["dest_lib"]=dest
for sub in ["config","samplers","models","datasets","runner"]:
    try: m=__import__(f"dest.{sub}", fromlist=[sub]); sys.modules[f"dest_lib.{sub}"]=m
    except: pass
print("✅ DEST instalado (fix Collatz 98% dup → 45k únicos)")
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 1. Config Batch ablation
from dest_lib.config import get_config
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["stochastic","collatz_v3"]
config["seeds"]=[200,201,202,203,204]
config["epochs"]=15
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["val_fraction"]=0.1
config["verbose"]=True
batches=[64,128,256]
print(f"Batches: {batches} × 2 samplers × {len(config['seeds'])} seeds = {len(batches)*2*len(config['seeds'])} runs")


In [ ]:
# 2. Ejecutar batch × sampler × seed
import os, time, json
from dest_lib.runner import ExperimentRunner
total=len(batches)*len(config["samplers"])*len(config["seeds"])
done=0
for bs in batches:
    config["batch_size"]=bs
    config["output_dir"]=f"./dest_batch_{bs}"
    runner=ExperimentRunner(config)
    for seed in config["seeds"]:
        for sampler_name in config["samplers"]:
            exp_id=f"CIFAR10_{sampler_name}_bs{bs}"
            out_file=os.path.join(config["output_dir"], f"{exp_id}_{sampler_name}_seed_{seed}.json")
            if os.path.exists(out_file):
                try:
                    j=json.load(open(out_file))
                    if j.get("status")=="COMPLETE" and len(j.get("test_accs",[]))==15:
                        print(f"⏭️ bs{bs} {sampler_name} {seed} ya completo"); done+=1; continue
                    else: os.remove(out_file)
                except: os.remove(out_file) if os.path.exists(out_file) else None
            print(f"\n[{done+1}/{total}] bs={bs} {sampler_name} seed {seed}")
            r=runner.run_single_seed(exp_id=exp_id, sampler_name=sampler_name, seed=seed, dataset="CIFAR10")
            print(f"✅ bs{bs} {sampler_name} {seed}: {r.final_test_acc:.2f}%")
            done+=1


In [ ]:
# 3. Resumen tabla batch vs sampler
import glob, json, numpy as np
from collections import defaultdict
for bs in [64,128,256]:
    pattern=f"dest_batch_{bs}/*.json"
    files=[f for f in glob.glob(pattern) if "sampler_name" in json.load(open(f))]
    print(f"\nBatch {bs}: {len(files)} JSONs")
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j["sampler_name"]].append(j["final_test_acc"])
    for s in ["stochastic","collatz_v3"]:
        arr=groups[s]
        if arr: print(f"  {s:12s}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f}")
    if "stochastic" in groups and "collatz_v3" in groups:
        import scipy.stats as stats
        diff=np.array(groups["collatz_v3"])-np.array(groups["stochastic"])
        t,p=stats.ttest_rel(groups["collatz_v3"], groups["stochastic"])
        print(f"  diff V3-stoch: {np.mean(diff):+.2f} p={p:.4f}")
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
for sampler in ["stochastic","collatz_v3"]:
    means=[]; stds=[]
    for bs in [64,128,256]:
        files=[f for f in glob.glob(f"dest_batch_{bs}/*.json") if json.load(open(f)).get("sampler_name")==sampler]
        arr=[json.load(open(f))["final_test_acc"] for f in files]
        means.append(np.mean(arr) if arr else 0)
        stds.append(np.std(arr,ddof=1) if len(arr)>1 else 0)
    plt.errorbar([64,128,256], means, yerr=stds, marker='o', label=sampler, capsize=4)
plt.xlabel("Batch size"); plt.ylabel("Test acc %"); plt.title("Batch ablation (CIFAR-10)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("batch_ablation.png", dpi=200)
plt.show()


In [ ]:
# Zip y descarga
import shutil, os, glob, json
files=[f for f in glob.glob("dest_*/*.json") if "sampler_name" in json.load(open(f))]
print(f"JSONs válidos: {len(files)}")
shutil.make_archive("resultados_"+title_md.split()[1],"zip","dest_"+title_md.split()[1].lower() if "Barrido" in title_md else "dest_batch")
print("ZIP listo")
from google.colab import files; files.download(glob.glob("*.zip")[0])
